# Evaluating Random Forest

In this final exercise you'll be evaluating the results of cross-validation on a Random Forest model.

The following have already been created:

- `cv` - a cross-validator which has already been fit to the training data
- `evaluator` — a `BinaryClassificationEvaluator` object and
- `flights_test` — the testing data.

## Instructions

- Print a list of average AUC metrics across all models in the parameter grid.
- Display the average AUC for the best model. This will be the largest AUC in the list.
- Print an explanation of the `maxDepth` and `featureSubsetStrategy` parameters for the best model.
- Display the AUC for the best model predictions on the testing data.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flight_manipulate_columns').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [3]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [7]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M4-EnsemblesAndPipelines/4_Ensembles/dataset/flights.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')


flights = flights.drop('flight')
flights = flights.dropna()

from pyspark.sql.functions import round
flights = flights.withColumn('km', round(flights.mile * 1.60934, 0))\
                .drop('mile')\
				.withColumn('label', (flights.delay > 15).cast('integer'))

#flights = flights.sample(0.25, seed=13)

from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=[
    'mon', 'depart', 'duration'
	], outputCol='features')
flights = assembler.transform(flights)
flights = flights.select('mon', 'depart', 'duration', 'features', 'label')

flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=17)

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [12]:
# Create a random forest classifier
forest = RandomForestClassifier()

# Create a parameter grid
params = ParamGridBuilder() \
            .addGrid(forest.featureSubsetStrategy, ['all', 'onethird', 'sqrt', 'log2']) \
            .addGrid(forest.maxDepth, [2, 5, 10]) \
            .build()

# Create a binary classification evaluator
evaluator = BinaryClassificationEvaluator()

# Create a cross-validator
cv = CrossValidator(estimator = forest\
                    , estimatorParamMaps = params, evaluator = evaluator, numFolds = 5)
cv = cv.fit(flights_train)

In [ ]:
# from os import system
# import pickle

# # system('unzip ./flights-random-forest-cv.zip')
# from pyspark.ml.tuning import CrossValidatorModel
# # cv = CrossValidatorModel.load('./flights-random-forest-cv')
# cv = CrossValidatorModel.load('E:\\Spark_Hadoop_Windows\\C-MachineLearningWithPySpark\\M4\\4_Ensembles\\flights-random-forest-cv')

# cv.avgMetrics = pickle.load(open('./flights-random-forest-cv-metrics.pickle', 'rb'))

# # from pyspark.ml.evaluation import BinaryClassificationEvaluator
# evaluator = BinaryClassificationEvaluator()

In [ ]:
# # Run this paragraph on Ubuntu instead of above

# from os import system
# import pickle

# # system('unzip ./flights-random-forest-cv.zip')
# from pyspark.ml.tuning import CrossValidatorModel
# # cv = CrossValidatorModel.load('./flights-random-forest-cv')
# cv = CrossValidatorModel.load('file:///home/talentum/test-jupyter/C-MachineLearningWithPySpark/M4/4_Ensembles/flights-random-forest-cv')

# cv.avgMetrics = pickle.load(open('file:///home/talentum/test-jupyter/C-MachineLearningWithPySpark/M4/4_Ensembles/flights-random-forest-cv-metrics.pickle', 'rb'))

# # from pyspark.ml.evaluation import BinaryClassificationEvaluator
# evaluator = BinaryClassificationEvaluator()

In [ ]:
# Average AUC for each parameter combination in grid
print(cv.____)

# Average AUC for the best model
print(____(____))

# What's the optimal parameter value for maxDepth?
print(cv.____.explainParam('____'))
# What's the optimal parameter value for featureSubsetStrategy?
print(cv.____.____(____))

# AUC for best model on testing data
print(evaluator.____(____.____(____)))

In [13]:
# Average AUC for each parameter combination in grid
print(cv.avgMetrics)

# Average AUC for the best model
print(max(cv.avgMetrics))

# What's the optimal parameter value for maxDepth?
print(cv.bestModel.explainParam('maxDepth'))
# What's the optimal parameter value for featureSubsetStrategy?
print(cv.bestModel.explainParam('featureSubsetStrategy'))

# AUC for best model on testing data
print(evaluator.evaluate(cv.transform(flights_test)))

[0.6170747310663454, 0.6622608452519864, 0.6704117628388464, 0.6443659739147026, 0.6619679778660235, 0.6740537668992048, 0.6426363543402077, 0.6640773548115573, 0.6723973806326582, 0.6426363543402077, 0.6640773548115573, 0.6723973806326582]
0.6740537668992048
maxDepth: Maximum depth of the tree. (Nonnegative) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. (default: 5, current: 10)
featureSubsetStrategy: The number of features to consider for splits at each tree node. Supported options: auto, all, onethird, sqrt, log2, (0.0-1.0], [1-n]. (default: auto, current: onethird)
0.6754594088066869


Fantastic! Optimized Random Forest > Random Forest > Decision Tree

<script.py> output:
    [0.61550451929848, 0.661275302749083, 0.6832959983649716, 0.6790399103856084, 0.6404890400309002, 0.6659871420567183, 0.6808977119243277, 0.6867946590518151, 0.6414270561540629, 0.6653385916148042, 0.6832494433718275, 0.6851695159338953, 0.6414270561540629, 0.6653385916148042, 0.6832494433718275, 0.6851695159338953]
    0.6867946590518151
    maxDepth: Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30]. (default: 5, current: 20)
    featureSubsetStrategy: The number of features to consider for splits at each tree node. Supported options: 'auto' (choose automatically for task: If numTrees == 1, set to 'all'. If numTrees > 1 (forest), set to 'sqrt' for classification and to 'onethird' for regression), 'all' (use all features), 'onethird' (use 1/3 of the features), 'sqrt' (use sqrt(number of features)), 'log2' (use log2(number of features)), 'n' (when n is in the range (0, 1.0], use n * number of features. When n is in the range (1, number of features), use n features). default = 'auto' (default: auto, current: onethird)
    0.7026685175668